# Figure generation for "The Care Gap in the Kingdom"

Produces all chart artifacts saved to `../output/`:
- `fig1.html` — before/after drive-time comparison maps
- `fig2.html` — population by block group with underserved boundary
- `fig3.html` — interactive placement explorer
- `fig4.html` — ranked placement table
- `placement_grids/{osm_id}.png` — per-site drive-time heatmaps

In [1]:
from __future__ import annotations

import base64
import io
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd
import requests
import shapely
import shapely.geometry as sg
from PIL import Image
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
from shapely.ops import unary_union

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
GRIDS_DIR = OUTPUT_DIR / "placement_grids"
OUTPUT_DIR.mkdir(exist_ok=True)
GRIDS_DIR.mkdir(exist_ok=True)

OSRM_BASE = "http://127.0.0.1:8008"
MAX_BATCH = 99        # addresses per OSRM table request (99 + 1 destination = 100 coords)
REQUEST_TIMEOUT_S = 30
OSRM_WORKERS = 16

GRID_RES = 250    # pixels per axis for drive-time heatmaps

# Drive-time color ramp: green → yellow → red
CMAP_COLORS = ["#2ecc71", "#82e0aa", "#f9e79f", "#f0b27a", "#e74c3c", "#8b0000"]
CMAP_BREAKS = np.array([0, 10, 15, 20, 30, 45, 60], dtype=float)
CMAP_NORM   = np.linspace(0, 1, len(CMAP_BREAKS))

# Approximate coordinates of the closed Hardwick Walgreens (65 N Main St)
HARDWICK_WALGREENS = {"name": "Walgreens Hardwick (closed 2024)", "lat": 44.5044, "lon": -72.3660}

In [2]:
# ── Load base data ───────────────────────────────────────────────────────────

addresses  = pd.read_csv(DATA_DIR / "addresses_with_population_weights.csv")
dur_raw    = np.load(DATA_DIR / "osrm_address_pharmacy_duration_sec.npy")
with (DATA_DIR / "pharmacies_active.json").open() as f:
    pharm_records = json.load(f)
boundary_gdf = gpd.read_file(DATA_DIR / "counties_five_vt.geojson")
sites_df = pd.read_csv(DATA_DIR / "placement_hypothetical_sites.csv")

# ── Filter to valid rows ──────────────────────────────────────────────────────

w_raw = addresses["population_weight"].to_numpy(dtype=np.float64)
d_sec = np.where(np.isfinite(dur_raw), dur_raw.astype(np.float64), np.inf)
t_nearest_sec = d_sec.min(axis=1)
t_min = t_nearest_sec / 60.0

keep = np.isfinite(t_min) & np.isfinite(w_raw) & (w_raw >= 0)
addresses = addresses.loc[keep].reset_index(drop=True)
w     = w_raw[keep]
d_sec = d_sec[keep]
t_min = t_min[keep]

addr_lon = addresses["lon"].to_numpy(dtype=np.float64)
addr_lat = addresses["lat"].to_numpy(dtype=np.float64)

# Marginal loss per existing pharmacy (second-best rerouting)
argmin = np.argmin(d_sec, axis=1)
d_part = np.partition(d_sec, 1, axis=1)[:, :2]
t1_sec = d_part.min(axis=1)
t2_sec = d_part.max(axis=1)

# ── Region geometry ───────────────────────────────────────────────────────────

region = unary_union(boundary_gdf.geometry)
minx, miny, maxx, maxy = region.bounds
center_lat = float(addr_lat.mean())
center_lon = float(addr_lon.mean())

# ── Pharmacy inside/outside study region ─────────────────────────────────────

pharm_pts    = shapely.points([p["lon"] for p in pharm_records], [p["lat"] for p in pharm_records])
pharm_inside = shapely.within(pharm_pts, region).tolist()   # bool list, same order as pharm_records

# ── Town lookup: nearest addresses → most common USPS city ───────────────────

from collections import Counter as _Counter

_town_tree = cKDTree(np.column_stack([addr_lat, addr_lon]))   # note: (lat, lon) for query

def nearest_town(lat: float, lon: float, k: int = 40) -> str:
    """Return the most common USPS city among the k nearest sampled addresses."""
    _, idx = _town_tree.query([lat, lon], k=k)
    cities = addresses.iloc[idx]["city"].dropna()
    if cities.empty:
        return ""
    return _Counter(cities).most_common(1)[0][0].title()

print(f"Addresses: {len(addresses):,}  |  Pharmacies: {len(pharm_records)} ({sum(pharm_inside)} inside boundary)  |  Sites: {len(sites_df)}")
print(f"Median drive time: {np.median(t_min):.1f} min  |  Max: {t_min.max():.1f} min")

Addresses: 86,672  |  Pharmacies: 224 (30 inside boundary)  |  Sites: 237
Median drive time: 11.9 min  |  Max: 111.4 min


In [3]:
# ── Census block choropleth data ──────────────────────────────────────────────
# Block geometries come from Census TIGER 2020PL (geometry only, no population);
# block populations are fetched separately from the 2020 Decennial Census API.
# All artifacts are cached so subsequent runs are instant.

import urllib.request
import zipfile
import tempfile

STUDY_COUNTY_FIPS = ["005", "009", "015", "019", "023"]   # Caledonia, Essex, Lamoille, Orleans, Washington
STUDY_PREFIXES    = {"50" + f for f in STUDY_COUNTY_FIPS}

GEOM_CACHE  = DATA_DIR / "vt_blocks_geom.geojson"
POP_CACHE   = DATA_DIR / "vt_blocks_pop.csv"
BLOCK_CACHE = DATA_DIR / "vt_blocks.geojson"

if not BLOCK_CACHE.exists():
    # ── block geometries ──────────────────────────────────────────────────────
    if not GEOM_CACHE.exists():
        print("Downloading VT 2020 block geometries from Census TIGER (~34 MB)…")
        url = "https://www2.census.gov/geo/tiger/TIGER2020PL/STATE/50_VERMONT/50/tl_2020_50_tabblock20.zip"
        with urllib.request.urlopen(url) as resp:
            raw = resp.read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            with tempfile.TemporaryDirectory() as tmpdir:
                z.extractall(tmpdir)
                shp = next(Path(tmpdir).glob("*.shp"))
                blocks_all = gpd.read_file(str(shp)).to_crs("EPSG:4326")
        blocks_geom = blocks_all[
            blocks_all["GEOID20"].str[:5].isin(STUDY_PREFIXES)
        ][["GEOID20", "geometry"]].copy()
        blocks_geom.to_file(str(GEOM_CACHE), driver="GeoJSON")
        print(f"Saved {len(blocks_geom):,} study-region block geometries → {GEOM_CACHE}")
    else:
        blocks_geom = gpd.read_file(str(GEOM_CACHE))
        print(f"Loaded {len(blocks_geom):,} block geometries from cache")

    # ── block populations from 2020 Decennial Census API ─────────────────────
    if not POP_CACHE.exists():
        print("Fetching block populations from 2020 Decennial Census API…")
        pop_dfs = []
        for county in STUDY_COUNTY_FIPS:
            url = (
                "https://api.census.gov/data/2020/dec/pl"
                f"?get=P1_001N&for=block:*&in=state:50+county:{county}"
            )
            resp = requests.get(url, timeout=60)
            rows = resp.json()
            df = pd.DataFrame(rows[1:], columns=rows[0])
            df["GEOID20"] = df["state"] + df["county"] + df["tract"] + df["block"]
            df["POP20"]   = df["P1_001N"].astype(int)
            pop_dfs.append(df[["GEOID20", "POP20"]])
            print(f"  County {county}: {len(df):,} blocks")
        block_pop = pd.concat(pop_dfs, ignore_index=True)
        block_pop.to_csv(str(POP_CACHE), index=False)
        print(f"Saved {len(block_pop):,} population records → {POP_CACHE}")
    else:
        block_pop = pd.read_csv(str(POP_CACHE))
        print(f"Loaded {len(block_pop):,} block populations from cache")

    # ── join, keep only populated blocks, simplify ────────────────────────────
    blocks = blocks_geom.merge(block_pop, on="GEOID20", how="inner")
    blocks = blocks[blocks["POP20"] > 0].copy()
    blocks["geometry"] = blocks["geometry"].simplify(0.0005, preserve_topology=True)
    blocks[["GEOID20", "POP20", "geometry"]].to_file(str(BLOCK_CACHE), driver="GeoJSON")
    print(f"Saved {len(blocks):,} populated blocks → {BLOCK_CACHE}")
else:
    blocks = gpd.read_file(str(BLOCK_CACHE))
    print(f"Loaded {len(blocks):,} populated blocks from cache")

import geobuf as _geobuf

ASSETS_DIR = OUTPUT_DIR / "assets"
ASSETS_DIR.mkdir(exist_ok=True)

# ── Quintile breaks for 5-class Blues choropleth ──────────────────────────────
pops = blocks["POP20"].values
pop_breaks_arr = [int(np.percentile(pops, p)) for p in (20, 40, 60, 80)]
BG_COLORS = ["#c6dbef", "#9ecae1", "#6baed6", "#3182bd", "#08519c"]

def _quintile(pop: int) -> int:
    if pop > pop_breaks_arr[3]: return 4
    if pop > pop_breaks_arr[2]: return 3
    if pop > pop_breaks_arr[1]: return 2
    if pop > pop_breaks_arr[0]: return 1
    return 0

# Encode with pre-computed quintile index — no raw population counts needed
_block_feats = json.loads(blocks[["geometry"]].to_json())
for feat, pop_val in zip(_block_feats["features"], blocks["POP20"]):
    feat["properties"] = {"q": _quintile(int(pop_val))}

BLOCKS_PBF = ASSETS_DIR / "blocks.pbf"
BLOCKS_PBF.write_bytes(_geobuf.encode(_block_feats))
pbf_kb = BLOCKS_PBF.stat().st_size / 1024
print(f"blocks.pbf: {pbf_kb:.0f} KB — {len(_block_feats['features']):,} features, quintile breaks: {pop_breaks_arr}")

BG_LABELS_JS = json.dumps([
    f"< {pop_breaks_arr[0]:,}",
    f"{pop_breaks_arr[0]:,}–{pop_breaks_arr[1]:,}",
    f"{pop_breaks_arr[1]:,}–{pop_breaks_arr[2]:,}",
    f"{pop_breaks_arr[2]:,}–{pop_breaks_arr[3]:,}",
    f"> {pop_breaks_arr[3]:,}",
])

Loaded 5,187 populated blocks from cache


blocks.pbf: 345 KB — 5,187 features, quintile breaks: [6, 12, 21, 40]


In [4]:
# ── Grid and interpolation setup (computed once) ─────────────────────────────

lon_g = np.linspace(minx, maxx, GRID_RES)
lat_g = np.linspace(miny, maxy, GRID_RES)
LON_MESH, LAT_MESH = np.meshgrid(lon_g, lat_g)

# Region mask on grid
_gpts = shapely.points(LON_MESH.ravel(), LAT_MESH.ravel())
INSIDE_MASK = shapely.within(_gpts, region).reshape(LON_MESH.shape)

# Precompute Delaunay triangulation once (reused for all LinearTriInterpolator calls)
TRI = mtri.Triangulation(addr_lon, addr_lat)

# Precompute nearest-neighbour indices from grid to address points (fill extrapolation gaps)
_kdtree = cKDTree(np.column_stack([addr_lon, addr_lat]))
_flat   = np.column_stack([LON_MESH.ravel(), LAT_MESH.ravel()])
_NN_IDX = _kdtree.query(_flat)[1]          # shape (GRID_RES²,)

def nn_on_grid(values: np.ndarray) -> np.ndarray:
    """Nearest-neighbour lookup: address values → grid, shape (GRID_RES, GRID_RES)."""
    return values[_NN_IDX].reshape(LON_MESH.shape)

def interp_to_grid(values: np.ndarray) -> np.ndarray:
    """
    Interpolate address-level values to (GRID_RES, GRID_RES) grid.
    Uses precomputed Delaunay triangulation; NaN outside convex hull filled by NN.
    Outside the study region is set to NaN.
    """
    lin = mtri.LinearTriInterpolator(TRI, values)
    gz  = lin(LON_MESH, LAT_MESH).filled(np.nan)
    gz  = np.where(np.isnan(gz), nn_on_grid(values), gz)
    gz  = np.where(INSIDE_MASK, gz, np.nan)
    return gz

print("Grid, triangulation, and NN index ready.")

Grid, triangulation, and NN index ready.


In [5]:
# ── Drive-time colormap ───────────────────────────────────────────────────────

def _build_drive_cmap() -> mcolors.LinearSegmentedColormap:
    rgba = [mcolors.to_rgba(c) for c in CMAP_COLORS]
    pos  = np.linspace(0, 1, len(CMAP_COLORS))
    cdict = {
        ch: [(pos[i], rgba[i][j], rgba[i][j]) for i in range(len(rgba))]
        for j, ch in enumerate(["red", "green", "blue"])
    }
    return mcolors.LinearSegmentedColormap("drive_time", cdict)

DRIVE_CMAP = _build_drive_cmap()


def drive_time_rgba(t_minutes: np.ndarray, opacity: float = 0.7) -> np.ndarray:
    """Address-level drive times → RGBA float32 grid (GRID_RES, GRID_RES, 4), flipped for Leaflet."""
    gz     = interp_to_grid(t_minutes)
    normed = np.interp(np.clip(gz, 0, 60), CMAP_BREAKS, CMAP_NORM)
    rgba   = DRIVE_CMAP(normed).astype(np.float32)
    rgba[..., 3] = np.where(np.isnan(gz), 0.0, opacity)
    return np.flipud(rgba)


def rgba_to_png_bytes(rgba: np.ndarray) -> bytes:
    img = Image.fromarray((rgba * 255).astype(np.uint8), mode="RGBA")
    buf = io.BytesIO()
    img.save(buf, format="PNG", optimize=True)
    return buf.getvalue()


def rgba_to_b64png(rgba: np.ndarray) -> str:
    return "data:image/png;base64," + base64.b64encode(rgba_to_png_bytes(rgba)).decode()


def save_rgba_png(rgba: np.ndarray, path: Path) -> None:
    path.write_bytes(rgba_to_png_bytes(rgba))


_CONTOUR_SIMPLIFY_TOL = 0.003   # ~300 m in degrees; keeps contour file size manageable


def contour_geojson(t_minutes: np.ndarray, threshold: float = 20.0) -> dict:
    """
    GeoJSON FeatureCollection of areas where t_minutes > threshold.
    Uses matplotlib contourf + allsegs (avoids CLOSEPOLY path-code issues in mpl ≥3.8).
    Output polygons are simplified at ~300 m tolerance to keep file sizes small.
    """
    gz  = interp_to_grid(t_minutes)
    gz2 = np.where(INSIDE_MASK & ~np.isnan(gz), gz, 0.0)

    if gz2.max() <= threshold:
        return {"type": "FeatureCollection", "features": []}

    fig, ax = plt.subplots(figsize=(4, 4))
    cs = ax.contourf(LON_MESH, LAT_MESH, gz2, levels=[threshold, gz2.max() + 1.0])
    plt.close(fig)

    # allsegs[0] is the list of closed segment arrays for the single level interval
    polys: list[sg.Polygon] = []
    for seg in cs.allsegs[0]:
        if len(seg) >= 3:
            try:
                p = sg.Polygon(seg).buffer(0)   # buffer(0) fixes minor self-intersections
                if p.is_valid and not p.is_empty:
                    polys.append(p)
            except Exception:
                pass

    if not polys:
        return {"type": "FeatureCollection", "features": []}

    combined = unary_union(polys).intersection(region)
    if combined.is_empty:
        return {"type": "FeatureCollection", "features": []}

    simplified = combined.simplify(_CONTOUR_SIMPLIFY_TOL, preserve_topology=True)
    return {
        "type": "FeatureCollection",
        "features": [{"type": "Feature", "properties": {}, "geometry": sg.mapping(simplified)}],
    }

print("Rendering helpers ready.")

Rendering helpers ready.


In [6]:
# ── OSRM: fetch drive times from all addresses to one destination ─────────────

def fetch_osrm_column(lon: float, lat: float) -> np.ndarray:
    """
    Drive times (seconds) from every address to a single destination point.
    Batches 99 addresses + 1 destination per OSRM table request.
    Uses a thread pool for concurrent requests.
    Returns float64 array, length = len(addr_lon); np.inf where no route.
    """
    n     = len(addr_lon)
    times = np.full(n, np.inf, dtype=np.float64)

    def _batch(i0: int) -> tuple[int, list]:
        i1    = min(n, i0 + MAX_BATCH)
        n_src = i1 - i0
        coords = (
            ";".join(f"{addr_lon[j]},{addr_lat[j]}" for j in range(i0, i1))
            + f";{lon},{lat}"
        )
        sources = ";".join(str(k) for k in range(n_src))
        url = (
            f"{OSRM_BASE}/table/v1/driving/{coords}"
            f"?sources={sources}&destinations={n_src}&annotations=duration"
        )
        try:
            resp = requests.get(url, timeout=REQUEST_TIMEOUT_S)
            data = resp.json()
            if data.get("code") == "Ok":
                return i0, [row[0] if row else None for row in data["durations"]]
        except Exception:
            pass
        return i0, [None] * n_src

    with ThreadPoolExecutor(max_workers=OSRM_WORKERS) as pool:
        for i0, vals in pool.map(_batch, range(0, n, MAX_BATCH)):
            for k, v in enumerate(vals):
                if v is not None:
                    times[i0 + k] = float(v)

    return times

print("OSRM helper ready.")

OSRM helper ready.


## FIG 1: Before / After — Hardwick Walgreens closure

Two synchronized maps showing drive times **before** (with Hardwick Walgreens) and **after** (current state).
The before-state is simulated by adding the closed pharmacy back to the matrix via OSRM.

In [7]:
WALGREENS_CACHE = DATA_DIR / "hardwick_walgreens_drive_times_sec.npy"

if WALGREENS_CACHE.exists():
    t_walgreens_sec = np.load(WALGREENS_CACHE)
    print(f"Loaded cached Walgreens times ({len(t_walgreens_sec):,} addresses)")
else:
    print("Fetching Hardwick Walgreens drive times via OSRM …")
    t_walgreens_sec = fetch_osrm_column(HARDWICK_WALGREENS["lon"], HARDWICK_WALGREENS["lat"])
    np.save(WALGREENS_CACHE, t_walgreens_sec)
    print(f"Saved → {WALGREENS_CACHE}")

t_before_min = np.minimum(t_min, t_walgreens_sec / 60.0)

saved = float(np.sum(w * np.maximum(0.0, t_min - t_before_min)))
pct_over20_before = float(np.sum(w[t_before_min > 20]) / w.sum() * 100)
pct_over20_after  = float(np.sum(w[t_min > 20]) / w.sum() * 100)

print(f"Person-minutes saved by Hardwick Walgreens: {saved:,.0f}")
print(f"% population >20 min  —  before: {pct_over20_before:.1f}%  |  after: {pct_over20_after:.1f}%")

Loaded cached Walgreens times (86,672 addresses)
Person-minutes saved by Hardwick Walgreens: 110,155
% population >20 min  —  before: 14.1%  |  after: 17.7%


In [8]:
before_b64 = rgba_to_b64png(drive_time_rgba(t_before_min))
after_b64  = rgba_to_b64png(drive_time_rgba(t_min))

_pharm_json   = json.dumps([
    {"lat": p["lat"], "lon": p["lon"], "name": p["name"], "city": p["city"], "inside": bool(pharm_inside[i])}
    for i, p in enumerate(pharm_records)
])
_county_json  = boundary_gdf.to_json()
_bounds_json  = json.dumps([[miny, minx], [maxy, maxx]])
_center_json  = json.dumps([center_lat, center_lon])
_walgreens_js = json.dumps(HARDWICK_WALGREENS)

_legend_items = "\n".join(
    f'<span style="display:inline-block;width:14px;height:14px;background:{c};margin-right:4px;border-radius:2px"></span>'
    f'{"<" if i == 0 else ""}{int(CMAP_BREAKS[min(i+1, len(CMAP_BREAKS)-1)])} min'
    for i, c in enumerate(CMAP_COLORS)
)

FIG1_HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Before / After: Hardwick Walgreens</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
<script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:system-ui,-apple-system,sans-serif;background:#f5f5f0;display:flex;flex-direction:column;height:100vh}}
  #header{{padding:10px 16px 6px;background:#fff;border-bottom:1px solid #ddd;flex-shrink:0}}
  #header h2{{font-size:.95rem;color:#444;font-weight:600;margin-bottom:4px}}
  #legend{{display:flex;flex-wrap:wrap;gap:8px;font-size:.75rem;color:#555;align-items:center}}
  #maps{{display:flex;flex:1;min-height:0}}
  .pane{{flex:1;display:flex;flex-direction:column;min-width:0}}
  .pane-label{{text-align:center;padding:5px 8px;font-size:.8rem;font-weight:600;background:#fff;border-bottom:1px solid #e0e0e0;color:#333;flex-shrink:0}}
  .pane-label.before{{border-bottom-color:#27ae60}}
  .pane-label.after {{border-bottom-color:#c0392b}}
  .map{{flex:1}}
  #divider{{width:3px;background:#ddd;flex-shrink:0}}
</style>
</head>
<body>
<div id="header">
  <h2>Drive time to nearest pharmacy — before and after the Hardwick Walgreens closure</h2>
  <div id="legend">{_legend_items}</div>
</div>
<div id="maps">
  <div class="pane">
    <div class="pane-label before">Before closure</div>
    <div id="map-before" class="map"></div>
  </div>
  <div id="divider"></div>
  <div class="pane">
    <div class="pane-label after">After closure (2024 – present)</div>
    <div id="map-after" class="map"></div>
  </div>
</div>
<script>
const PHARMACIES = {_pharm_json};
const COUNTIES   = {_county_json};
const BOUNDS     = {_bounds_json};
const CENTER     = {_center_json};
const WG         = {_walgreens_js};
const IMG_BEFORE = "{before_b64}";
const IMG_AFTER  = "{after_b64}";
const TILE       = "https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png";
const TILE_ATTR  = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com">CARTO</a>';

function buildMap(id, img, showWalgreens) {{
  const m = L.map(id, {{zoomControl: id === 'map-before'}}).setView(CENTER, 9);
  L.tileLayer(TILE, {{attribution: TILE_ATTR, subdomains: 'abcd', maxZoom: 19}}).addTo(m);
  L.geoJSON(COUNTIES, {{style: {{fillOpacity:0, color:'#666', weight:1.5}}}}).addTo(m);
  L.imageOverlay(img, BOUNDS, {{opacity:0.65, interactive:false, zIndex:10}}).addTo(m);
  PHARMACIES.forEach(p => {{
    const style = p.inside
      ? {{radius:5, color:'#1a5276', fill:true, fillColor:'#3498db', fillOpacity:.9, weight:1}}
      : {{radius:4, color:'#aaa',    fill:true, fillColor:'#ccc',    fillOpacity:.45, weight:1}};
    L.circleMarker([p.lat, p.lon], style)
      .bindTooltip(p.name + (p.city ? ' · ' + p.city : ''))
      .addTo(m);
  }});
  if (showWalgreens) {{
    L.circleMarker([WG.lat, WG.lon], {{radius:8, color:'#6e0000', fill:true, fillColor:'#c0392b', fillOpacity:.95, weight:2}})
      .bindTooltip(WG.name, {{permanent:false}})
      .addTo(m);
  }}
  return m;
}}

const mapB = buildMap('map-before', IMG_BEFORE, true);
const mapA = buildMap('map-after',  IMG_AFTER,  false);

// Synchronize pan / zoom
let _syncing = false;
function sync(src, tgt) {{
  src.on('moveend', () => {{
    if (_syncing) return;
    _syncing = true;
    tgt.setView(src.getCenter(), src.getZoom(), {{animate:false}});
    _syncing = false;
  }});
}}
sync(mapB, mapA);
sync(mapA, mapB);
</script>
</body>
</html>"""

out = OUTPUT_DIR / "fig1.html"
out.write_text(FIG1_HTML, encoding="utf-8")
print(f"Saved → {out}")

Saved → ../output/fig1.html


## FIG 2: Population density with underserved boundary

Population-weighted density heatmap.  The red boundary outlines areas currently
≥ 20 minutes from any pharmacy.

In [9]:
underserved = contour_geojson(t_min, threshold=20.0)
pop_under_w = float(np.sum(w[t_min > 20]))
pop_total_w = float(w.sum())

_pharm_json2  = json.dumps([
    {"lat": p["lat"], "lon": p["lon"], "name": p["name"], "city": p["city"], "inside": bool(pharm_inside[i])}
    for i, p in enumerate(pharm_records)
])
_county_json2 = boundary_gdf.to_json()

# Build legend HTML from Python-computed breaks (avoids JS template complexity)
_bg_legend = "".join(
    f'<span style="display:inline-block;width:12px;height:12px;background:{c};'
    f'margin-right:5px;border-radius:2px;vertical-align:middle"></span>{label}<br>'
    for c, label in zip(BG_COLORS, [
        f"&lt;&nbsp;{pop_breaks_arr[0]:,}",
        f"{pop_breaks_arr[0]:,}–{pop_breaks_arr[1]:,}",
        f"{pop_breaks_arr[1]:,}–{pop_breaks_arr[2]:,}",
        f"{pop_breaks_arr[2]:,}–{pop_breaks_arr[3]:,}",
        f"&gt;&nbsp;{pop_breaks_arr[3]:,}",
    ])
)

FIG2_HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Population by census block and pharmacy deserts</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
<script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:system-ui,-apple-system,sans-serif;background:#f5f5f0;display:flex;flex-direction:column;height:100vh}}
  #header{{padding:10px 16px 6px;background:#fff;border-bottom:1px solid #ddd;flex-shrink:0}}
  #header h2{{font-size:.95rem;color:#444;font-weight:600;margin-bottom:4px}}
  #stat{{font-size:.8rem;color:#666;margin-top:2px}}
  #map{{flex:1}}
  .legend-box{{background:rgba(255,255,255,.92);padding:8px 10px;border-radius:6px;font-size:.75rem;line-height:1.8;color:#444}}
</style>
</head>
<body>
<svg style="position:absolute;width:0;height:0;overflow:hidden" aria-hidden="true">
  <defs>
    <pattern id="desert-hatch" patternUnits="userSpaceOnUse" width="3" height="3">
      <circle cx="2" cy="2" r="0.5" fill="#c0392b" fill-opacity="1"/>
    </pattern>
  </defs>
</svg>
<div id="header">
  <h2>Population by census block and pharmacy deserts — Northeast Vermont</h2>
  <div id="stat">
    <strong>{pop_under_w:,.0f}</strong> residents ({pop_under_w/pop_total_w*100:.1f}% of region) currently &gt;20 minutes from a pharmacy.
    Red boundary = 20+ minute pharmacy desert.
  </div>
</div>
<div id="map"></div>
<script type="module">
import * as geobuf from 'https://esm.sh/geobuf@3.0.2';
import Pbf from 'https://esm.sh/pbf@3.2.1';
const PHARMACIES = {_pharm_json2};
const COUNTIES   = {_county_json2};
const CENTER     = {_center_json};
const BG_COLORS  = {json.dumps(BG_COLORS)};
const BG_LABELS  = {BG_LABELS_JS};
const CONTOUR    = {json.dumps(underserved)};
const TILE       = "https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png";
const TILE_ATTR  = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com">CARTO</a>';

const m = L.map('map').setView(CENTER, 9);
L.tileLayer(TILE, {{attribution: TILE_ATTR, subdomains:'abcd', maxZoom:19}}).addTo(m);
L.geoJSON(COUNTIES, {{style: {{fillOpacity:0, color:'#888', weight:1}}}}).addTo(m);
fetch('assets/blocks.pbf')
  .then(r => r.arrayBuffer())
  .then(buf => {{
    L.geoJSON(geobuf.decode(new Pbf(buf)), {{
      style: f => ({{fillColor: BG_COLORS[f.properties.q], fillOpacity:0.25, stroke: false}}),
      onEachFeature: (f, l) => l.bindTooltip(BG_LABELS[f.properties.q] + ' residents'),
    }}).addTo(m);
  }});
L.geoJSON(CONTOUR, {{style: {{color:'#c0392b', weight:1, fillColor:'url(#desert-hatch)', fillOpacity:1}}}}).addTo(m);
PHARMACIES.forEach(p => {{
  const style = p.inside
    ? {{radius:5, color:'#1a5276', fill:true, fillColor:'#3498db', fillOpacity:.9, weight:1}}
    : {{radius:4, color:'#aaa',    fill:true, fillColor:'#ccc',    fillOpacity:.45, weight:1}};
  L.circleMarker([p.lat, p.lon], style)
    .bindTooltip(p.name + (p.city ? ' · '+p.city:''))
    .addTo(m);
}});

// Legend
const legend = L.control({{position:'bottomright'}});
legend.onAdd = () => {{
  const d = L.DomUtil.create('div','legend-box');
  d.innerHTML = '<b>Residents per census block</b><br>{_bg_legend}'
    + '<span style="display:inline-block;width:14px;height:3px;background:#c0392b;margin-right:5px;vertical-align:middle"></span>&ge;20 min desert';
  return d;
}};
legend.addTo(m);
</script>
</body>
</html>"""

out2 = OUTPUT_DIR / "fig2.html"
out2.write_text(FIG2_HTML, encoding="utf-8")
print(f"Saved → {out2}")

Saved → ../output/fig2.html


## FIG 3: Interactive placement explorer

For each of the 237 hypothetical sites:
1. Fetch per-address drive times via OSRM (skipped if PNG already exists)
2. Compute new minimum times (`min(current, new_site)`)
3. Render drive-time heatmap PNG → `placement_grids/{osm_id}.png`
4. Compute 20-min contour GeoJSON (embedded in the interactive HTML)

First run takes ~10 minutes; subsequent runs skip cached PNGs.

In [10]:
from rich.progress import (
    BarColumn, Progress, TaskProgressColumn,
    TextColumn, TimeElapsedColumn, TimeRemainingColumn,
)

sites_df = pd.read_csv(DATA_DIR / "placement_hypothetical_sites.csv")

# Three cache artifacts per site: PNG heatmap, contour GeoJSON, drive-time array
# Having all three means fully cached; the .npy avoids re-fetching OSRM for regeneration
def _cached(osm_id: int) -> bool:
    return (
        (GRIDS_DIR / f"{osm_id}.png").exists()
        and (GRIDS_DIR / f"{osm_id}_contour.json").exists()
    )

def _has_tmin(osm_id: int) -> bool:
    return (GRIDS_DIR / f"{osm_id}_tmin.npy").exists()

todo = [row for _, row in sites_df.iterrows() if not _cached(int(row.osm_id))]
done = len(sites_df) - len(todo)
print(f"{done} / {len(sites_df)} sites fully cached — computing {len(todo)} remaining")

with Progress(
    TextColumn("[bold blue]{task.description}"),
    BarColumn(),
    TaskProgressColumn(),
    TimeElapsedColumn(),
    TimeRemainingColumn(),
) as prog:
    task = prog.add_task("Placement grids", total=len(todo))
    for row in todo:
        osm_id    = int(row.osm_id)
        png_path  = GRIDS_DIR / f"{osm_id}.png"
        cont_path = GRIDS_DIR / f"{osm_id}_contour.json"
        tmin_path = GRIDS_DIR / f"{osm_id}_tmin.npy"

        # Load cached drive times if available, else fetch from OSRM
        if tmin_path.exists():
            t_with = np.load(tmin_path)
        else:
            t_new_sec = fetch_osrm_column(float(row.lon), float(row.lat))
            t_with    = np.minimum(t_min, t_new_sec / 60.0).astype(np.float32)
            np.save(tmin_path, t_with)

        if not png_path.exists():
            save_rgba_png(drive_time_rgba(t_with), png_path)

        if not cont_path.exists():
            gj = contour_geojson(t_with, threshold=20.0)
            cont_path.write_text(json.dumps(gj))

        prog.advance(task)

n_png  = len(list(GRIDS_DIR.glob("*.png")))
n_cont = len(list(GRIDS_DIR.glob("*_contour.json")))
print(f"Done. {n_png} PNGs · {n_cont} contours")

Output()

237 / 237 sites fully cached — computing 0 remaining


Done. 237 PNGs · 237 contours


In [11]:
# ── Compute people_affected and town for every site ───────────────────────────

from rich.progress import Progress, BarColumn, TextColumn, TaskProgressColumn

print("Computing per-site people_affected and town …")
people_affected_map: dict[int, int] = {}
town_map: dict[int, str] = {}

with Progress(TextColumn("{task.description}"), BarColumn(), TaskProgressColumn()) as prog:
    task = prog.add_task("Loading site data", total=len(sites_df))
    for _, row in sites_df.iterrows():
        osm_id    = int(row.osm_id)
        tmin_path = GRIDS_DIR / f"{osm_id}_tmin.npy"
        if tmin_path.exists():
            t_with = np.load(tmin_path)
            # Strictly closer: t_with (float32) vs t_min (float64)
            people_affected_map[osm_id] = int(round(float(np.sum(w[t_with.astype(np.float64) < t_min - 0.01]))))
        else:
            people_affected_map[osm_id] = 0
        town_map[osm_id] = nearest_town(float(row.lat), float(row.lon))
        prog.advance(task)

print(f"Done. Example — Hardwick village: "
      f"people_affected={people_affected_map.get(158837472, '?')}, "
      f"town='{town_map.get(158837472, '?')}'")

# ── Build placement metadata (sorted by person_minutes_saved descending) ──────

placements_sorted = sites_df.sort_values("person_minutes_saved_one_way", ascending=False)

placements_js = json.dumps([
    {
        "osm_id":          int(r.osm_id),
        "name":            str(r.village),
        "place":           str(r.place),
        "lat":             float(r.lat),
        "lon":             float(r.lon),
        "minutes_saved":   float(r.person_minutes_saved_one_way),
        "people_affected": people_affected_map.get(int(r.osm_id), 0),
        "town":            town_map.get(int(r.osm_id), ""),
    }
    for _, r in placements_sorted.iterrows()
])

# ── Load contours from individual cached files ────────────────────────────────

contours: dict[int, dict] = {}
for cont_path in GRIDS_DIR.glob("*_contour.json"):
    osm_id = int(cont_path.stem.replace("_contour", ""))
    with cont_path.open() as f:
        contours[osm_id] = json.load(f)

missing_ct = [int(r.osm_id) for _, r in sites_df.iterrows() if int(r.osm_id) not in contours]
if missing_ct:
    print(f"WARNING: {len(missing_ct)} sites missing contour data — run pre-computation cell first")
else:
    print(f"All {len(contours)} contours loaded")

contours_js   = json.dumps({str(k): v for k, v in contours.items()})
_pharm_json3  = json.dumps([
    {"lat": p["lat"], "lon": p["lon"], "name": p["name"], "city": p["city"], "inside": bool(pharm_inside[i])}
    for i, p in enumerate(pharm_records)
])
_county_json3 = boundary_gdf.to_json()
top_osm_id    = int(placements_sorted.iloc[0].osm_id)

FIG3_HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Pharmacy placement explorer</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
<script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:system-ui,-apple-system,sans-serif;background:#f5f5f0;display:flex;flex-direction:column;height:100vh;overflow:hidden}}
  #header{{padding:8px 14px;background:#fff;border-bottom:1px solid #ddd;flex-shrink:0;display:flex;align-items:center;gap:16px;flex-wrap:wrap}}
  #header label{{font-size:.8rem;color:#666;white-space:nowrap}}
  #header select{{font-size:.82rem;padding:4px 8px;border:1px solid #ccc;border-radius:4px;max-width:260px;background:#fff}}
  #stat{{font-size:.85rem;color:#333;white-space:nowrap}}
  #stat strong{{color:#1a6b3c;font-size:1rem}}
  #maps{{display:flex;flex:1;min-height:0}}
  .pane{{flex:1;display:flex;flex-direction:column;min-width:0}}
  .pane-label{{text-align:center;padding:4px 8px;font-size:.78rem;font-weight:600;background:#fff;border-bottom:1px solid #e0e0e0;color:#555;flex-shrink:0}}
  .map{{flex:1}}
  #divider{{width:3px;background:#ddd;flex-shrink:0}}
  .village-marker-active {{stroke:#8b0000!important;stroke-width:3!important;fill:#e74c3c!important}}
</style>
</head>
<body>
<svg style="position:absolute;width:0;height:0;overflow:hidden" aria-hidden="true">
  <defs>
    <pattern id="desert-hatch" patternUnits="userSpaceOnUse" width="3" height="3">
      <circle cx="2" cy="2" r="0.5" fill="#c0392b" fill-opacity="1"/>
    </pattern>
  </defs>
</svg>
<div id="header">
  <label>Hypothetical pharmacy location:
    <select id="site-select"></select>
  </label>
  <div id="stat">
    <strong id="stat-val">—</strong> person·minutes saved &nbsp;·&nbsp;
    <strong id="stat-people">—</strong> people served
  </div>
</div>
<div id="maps">
  <div class="pane">
    <div class="pane-label">Drive-time with hypothetical pharmacy</div>
    <div id="map-left" class="map"></div>
  </div>
  <div id="divider"></div>
  <div class="pane">
    <div class="pane-label">Population by census block · red outline = 20+ minute pharmacy desert</div>
    <div id="map-right" class="map"></div>
  </div>
</div>
<script type="module">
import * as geobuf from 'https://esm.sh/geobuf@3.0.2';
import Pbf from 'https://esm.sh/pbf@3.2.1';
const PLACEMENTS = {placements_js};
const CONTOURS   = {contours_js};
const PHARMACIES = {_pharm_json3};
const COUNTIES   = {_county_json3};
const BOUNDS     = {_bounds_json};
const CENTER     = {_center_json};
const BG_COLORS  = {json.dumps(BG_COLORS)};
const BG_LABELS  = {BG_LABELS_JS};
const GRIDS_DIR  = "placement_grids/";
const TILE       = "https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png";
const TILE_ATTR  = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com">CARTO</a>';

// ── Build maps ────────────────────────────────────────────────────────────────
function makeBase(id, zoomCtrl) {{
  const m = L.map(id, {{zoomControl: zoomCtrl}}).setView(CENTER, 9);
  L.tileLayer(TILE, {{attribution: TILE_ATTR, subdomains:'abcd', maxZoom:19}}).addTo(m);
  L.geoJSON(COUNTIES, {{style: {{fillOpacity:0, color:'#888', weight:1}}}}).addTo(m);
  return m;
}}

const mapL = makeBase('map-left',  true);
const mapR = makeBase('map-right', false);

// Right-pane custom panes: bgPane below contour, both below markers (overlayPane=400)
mapR.createPane('contourPane');
mapR.getPane('contourPane').style.zIndex = 360;

// Block group choropleth — right pane only, loaded async
fetch('assets/blocks.pbf')
  .then(r => r.arrayBuffer())
  .then(buf => {{
    L.geoJSON(geobuf.decode(new Pbf(buf)), {{
      style: f => ({{fillColor: BG_COLORS[f.properties.q], fillOpacity:0.25, stroke: false}}),
      onEachFeature: (f, l) => l.bindTooltip(BG_LABELS[f.properties.q] + ' residents'),
    }}).addTo(mapR);
  }});

// Pharmacy markers on both maps
PHARMACIES.forEach(p => {{
  const opts = p.inside
    ? {{radius:5, color:'#1a5276', fill:true, fillColor:'#3498db', fillOpacity:.9, weight:1}}
    : {{radius:4, color:'#aaa',    fill:true, fillColor:'#ccc',    fillOpacity:.45, weight:1}};
  L.circleMarker([p.lat, p.lon], opts).bindTooltip(p.name+(p.city?' · '+p.city:'')).addTo(mapL);
  L.circleMarker([p.lat, p.lon], opts).bindTooltip(p.name+(p.city?' · '+p.city:'')).addTo(mapR);
}});

// ── Village markers (both maps) ───────────────────────────────────────────────
const markersL = {{}}, markersR = {{}};

PLACEMENTS.forEach(v => {{
  const base = {{radius:5, color:'#555', fill:true, fillColor:'#aaa', fillOpacity:.7, weight:1.5}};
  const mL = L.circleMarker([v.lat, v.lon], {{...base}}).bindTooltip(v.name).addTo(mapL);
  const mR = L.circleMarker([v.lat, v.lon], {{...base}}).bindTooltip(v.name).addTo(mapR);
  markersL[v.osm_id] = mL;
  markersR[v.osm_id] = mR;
  mL.on('click', () => selectSite(v.osm_id));
  mR.on('click', () => selectSite(v.osm_id));
}});

// ── Overlay state ─────────────────────────────────────────────────────────────
let heatLayer = null, contourLayer = null, activeId = null;

function selectSite(osm_id) {{
  const v = PLACEMENTS.find(p => p.osm_id === osm_id);
  if (!v) return;

  // Update stats
  document.getElementById('stat-val').textContent    = Math.round(v.minutes_saved).toLocaleString();
  document.getElementById('stat-people').textContent = Math.round(v.people_affected).toLocaleString();

  // Reset previous marker style
  if (activeId !== null) {{
    const dimStyle = {{radius:5, color:'#555', fillColor:'#aaa', fillOpacity:.6, weight:1.5}};
    markersL[activeId]?.setStyle(dimStyle);
    markersR[activeId]?.setStyle(dimStyle);
  }}

  // Highlight selected
  const activeStyle = {{radius:9, color:'#6e0000', fillColor:'#e74c3c', fillOpacity:.95, weight:2.5}};
  markersL[osm_id]?.setStyle(activeStyle).bringToFront();
  markersR[osm_id]?.setStyle(activeStyle).bringToFront();
  activeId = osm_id;

  // Swap heatmap (left map)
  if (heatLayer) {{ mapL.removeLayer(heatLayer); heatLayer = null; }}
  const src = GRIDS_DIR + osm_id + '.png';
  heatLayer = L.imageOverlay(src, BOUNDS, {{opacity:0.65, interactive:false, zIndex:10}});
  heatLayer.addTo(mapL);

  // Swap contour (right map)
  if (contourLayer) {{ mapR.removeLayer(contourLayer); contourLayer = null; }}
  const gj = CONTOURS[String(osm_id)];
  if (gj && gj.features && gj.features.length) {{
    contourLayer = L.geoJSON(gj, {{style: {{color:'#c0392b', weight:1, fillColor:'url(#desert-hatch)', fillOpacity:1}}, pane:'contourPane'}});
    contourLayer.addTo(mapR);
  }}

  // Sync dropdown
  document.getElementById('site-select').value = String(osm_id);
}}

// ── Dropdown ──────────────────────────────────────────────────────────────────
const sel = document.getElementById('site-select');
PLACEMENTS.sort((a, b) => a.name.localeCompare(b.name)).forEach(v => {{
  const opt = document.createElement('option');
  opt.value = String(v.osm_id);
  opt.textContent = v.town && v.town !== v.name ? v.name + ' (' + v.town + ')' : v.name;
  sel.appendChild(opt);
}});
sel.addEventListener('change', e => selectSite(parseInt(e.target.value)));

// ── Sync pan/zoom ─────────────────────────────────────────────────────────────
let _syncing = false;
function syncMaps(src, tgt) {{
  src.on('moveend', () => {{
    if (_syncing) return;
    _syncing = true;
    tgt.setView(src.getCenter(), src.getZoom(), {{animate:false}});
    _syncing = false;
  }});
}}
syncMaps(mapL, mapR);
syncMaps(mapR, mapL);

// ── Default selection ─────────────────────────────────────────────────────────
selectSite({top_osm_id});
</script>
</body>
</html>"""

out3 = OUTPUT_DIR / "fig3.html"
out3.write_text(FIG3_HTML, encoding="utf-8")
print(f"Saved → {out3}")
print(f"  (references PNG files in {GRIDS_DIR} via relative paths)")

Output()

Computing per-site people_affected and town …


Done. Example — Hardwick village: people_affected=8387, town='Hardwick'


All 237 contours loaded
Saved → ../output/fig3.html
  (references PNG files in ../output/placement_grids via relative paths)


## FIG 4: Placement rankings table

Sortable table of all 237 hypothetical sites by person-minutes saved.

In [12]:
table_data_js = json.dumps([
    {
        "rank":    i + 1,
        "name":    str(r.village),
        "town":    town_map.get(int(r.osm_id), ""),
        "minutes": int(round(r.person_minutes_saved_one_way)),
        "people":  people_affected_map.get(int(r.osm_id), 0),
    }
    for i, (_, r) in enumerate(placements_sorted.iterrows())
])

FIG4_HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Pharmacy placement rankings</title>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:system-ui,-apple-system,sans-serif;background:#f5f5f0;padding:24px 20px;color:#333}}
  h2{{font-size:1rem;font-weight:600;margin-bottom:8px;color:#222}}
  p.note{{font-size:.78rem;color:#777;margin-bottom:14px}}
  #search{{margin-bottom:10px;padding:6px 10px;border:1px solid #ccc;border-radius:4px;font-size:.82rem;width:240px}}
  table{{border-collapse:collapse;width:100%;max-width:780px;font-size:.82rem;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.1);border-radius:6px;overflow:hidden}}
  thead th{{background:#1a5276;color:#fff;padding:8px 12px;text-align:left;font-weight:600;cursor:pointer;user-select:none;white-space:nowrap}}
  thead th:hover{{background:#1a3e5c}}
  thead th span.arrow{{margin-left:4px;opacity:.6}}
  tbody tr:nth-child(even){{background:#f7fafd}}
  tbody tr:hover{{background:#ebf5fb}}
  td{{padding:7px 12px;border-bottom:1px solid #e8edf2}}
  td.rank{{color:#aaa;font-variant-numeric:tabular-nums;text-align:right;width:44px}}
  td.minutes{{font-variant-numeric:tabular-nums;text-align:right;width:140px;font-weight:600;color:#1a6b3c}}
  td.people{{font-variant-numeric:tabular-nums;text-align:right;width:120px;color:#555}}
  #count{{font-size:.75rem;color:#888;margin-top:8px}}
</style>
</head>
<body>
<h2>Hypothetical pharmacy sites — ranked by population-weighted time saved</h2>
<p class="note">
  Person·minutes saved (one-way) = sum of (current drive time − new drive time) × residents served,
  for every address that would be closer to the new location than their current nearest pharmacy.
</p>
<input id="search" type="text" placeholder="Filter by name…" oninput="renderTable()">
<table id="tbl">
  <thead>
    <tr>
      <th onclick="sortBy('rank')" title="Original rank">Rank<span class="arrow" id="arr-rank">↕</span></th>
      <th onclick="sortBy('name')" title="Village or hamlet name">Location<span class="arrow" id="arr-name">↕</span></th>
      <th onclick="sortBy('town')" title="Parent Vermont town">Town<span class="arrow" id="arr-town">↕</span></th>
      <th onclick="sortBy('minutes')" title="Person-minutes saved">Person·min saved<span class="arrow" id="arr-minutes">↕</span></th>
      <th onclick="sortBy('people')" title="People who gain a closer pharmacy">People served<span class="arrow" id="arr-people">↕</span></th>
    </tr>
  </thead>
  <tbody id="tbody"></tbody>
</table>
<div id="count"></div>
<script>
const DATA = {table_data_js};
let sortCol = 'rank', sortAsc = true;

function sortBy(col) {{
  if (sortCol === col) sortAsc = !sortAsc;
  else {{ sortCol = col; sortAsc = !['minutes','people'].includes(col); }}
  renderTable();
}}

function renderTable() {{
  const q = document.getElementById('search').value.toLowerCase();
  let rows = DATA.filter(r =>
    r.name.toLowerCase().includes(q) || r.town.toLowerCase().includes(q)
  );
  rows.sort((a, b) => {{
    const av = a[sortCol], bv = b[sortCol];
    const cmp = typeof av === 'string' ? av.localeCompare(bv) : av - bv;
    return sortAsc ? cmp : -cmp;
  }});
  ['rank','name','town','minutes','people'].forEach(c => {{
    const el = document.getElementById('arr-'+c);
    if (el) el.textContent = sortCol === c ? (sortAsc ? '↑' : '↓') : '↕';
  }});
  const tb = document.getElementById('tbody');
  tb.innerHTML = rows.map(r =>
    `<tr>
      <td class="rank">${{r.rank}}</td>
      <td>${{r.name}}</td>
      <td>${{r.town}}</td>
      <td class="minutes">${{r.minutes.toLocaleString()}}</td>
      <td class="people">${{r.people.toLocaleString()}}</td>
     </tr>`
  ).join('');
  document.getElementById('count').textContent = `Showing ${{rows.length}} of ${{DATA.length}} locations`;
}}

renderTable();
</script>
</body>
</html>"""

out4 = OUTPUT_DIR / "fig4.html"
out4.write_text(FIG4_HTML, encoding="utf-8")
print(f"Saved → {out4}")

Saved → ../output/fig4.html


In [13]:
print("─" * 50)
print("All outputs saved:")
for p in sorted(OUTPUT_DIR.glob("*.html")):
    print(f"  {p}")
print(f"  {GRIDS_DIR} ({len(list(GRIDS_DIR.glob('*.png')))} PNGs)")

──────────────────────────────────────────────────
All outputs saved:
  ../output/fig1.html
  ../output/fig2.html
  ../output/fig3.html
  ../output/fig4.html
  ../output/placement_grids (237 PNGs)
